In [12]:
from google.colab import drive
from pathlib import Path
import os
import sys

# Monta o Google Drive
drive.mount("/content/drive", force_remount=False)

# Pasta raiz do projeto
PROJECT_ROOT = Path("/content/drive/MyDrive/financeiro-pessoal")

# Garante que a pasta exista
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

# Define como diretório atual
os.chdir(PROJECT_ROOT)

# Adiciona ao PYTHONPATH
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("=" * 60)
print("Sistema Financeiro Pessoal")
print("=" * 60)
print(f"Projeto : {PROJECT_ROOT}")
print(f"Diretório atual : {Path.cwd()}")
print("=" * 60)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Sistema Financeiro Pessoal
Projeto : /content/drive/MyDrive/financeiro-pessoal
Diretório atual : /content/drive/MyDrive/financeiro-pessoal


In [ ]:
from tools.git import setup_git

setup_git(
    user_name="Daniel Cunha da Silva",
    user_email="danielcsilva1@yahoo.com.br"
)

In [63]:
from tools.git import status

status()

M Desenvolvimento.ipynb
 M src/domain/value_objects/__init__.py
 M src/domain/value_objects/percentage.py


In [64]:
from tools.git import commit

commit(
    "v0.1.0-alpha.7",
    "Implementa Value Object Percentage"
)

In [65]:
from tools.git import push

push()

In [62]:
FILES = {

"src/domain/value_objects/percentage.py": r'''
"""
Percentage Value Object.

Representa um percentual imutável.
"""

from __future__ import annotations

from dataclasses import dataclass
from decimal import Decimal, ROUND_HALF_UP
from typing import Union

Number = Union[int, float, Decimal]


@dataclass(frozen=True, slots=True)
class Percentage:

    value: Decimal

    def __post_init__(self) -> None:
        amount = self._to_decimal(self.value)
        object.__setattr__(
            self,
            "value",
            amount.quantize(
                Decimal("0.0001"),
                rounding=ROUND_HALF_UP,
            ),
        )

    @staticmethod
    def _to_decimal(value: Number | Decimal) -> Decimal:
        if isinstance(value, Decimal):
            return value
        return Decimal(str(value))

    @classmethod
    def zero(cls) -> "Percentage":
        return cls(Decimal("0"))

    @classmethod
    def from_factor(cls, factor: Number) -> "Percentage":
        return cls(cls._to_decimal(factor) * Decimal("100"))

    def to_decimal(self) -> Decimal:
        return self.value

    def factor(self) -> Decimal:
        return self.value / Decimal("100")

    def apply(self, amount) -> Decimal:
        return amount.to_decimal() * self.factor()

    def is_zero(self) -> bool:
        return self.value == Decimal("0")

    def is_positive(self) -> bool:
        return self.value > Decimal("0")

    def is_negative(self) -> bool:
        return self.value < Decimal("0")

    def __add__(self, other: "Percentage") -> "Percentage":
        if not isinstance(other, Percentage):
            return NotImplemented
        return Percentage(self.value + other.value)

    def __sub__(self, other: "Percentage") -> "Percentage":
        if not isinstance(other, Percentage):
            return NotImplemented
        return Percentage(self.value - other.value)

    def __mul__(self, other: Number) -> Decimal:
        return self.factor() * self._to_decimal(other)

    def __truediv__(self, other: Number) -> Decimal:
        return self.value / self._to_decimal(other)

    def __neg__(self) -> "Percentage":
        return Percentage(-self.value)

    def __abs__(self) -> "Percentage":
        return Percentage(abs(self.value))

    def __str__(self) -> str:
        return f"{self.value.normalize()}%"

    def __repr__(self) -> str:
        return f"Percentage({self.value})"
''',

"src/domain/value_objects/__init__.py": r'''
from .money import Money
from .percentage import Percentage

__all__ = [
    "Money",
    "Percentage",
]
'''

}

from tools.file_writer import write_files
write_files(FILES)

[OK] src/domain/value_objects/percentage.py
[OK] src/domain/value_objects/__init__.py


In [51]:
from src.domain.value_objects import Money

salario = Money("10500")

aluguel = Money("1800")

print(salario)
print(aluguel)
print(salario - aluguel)

R$ 10.500,00
R$ 1.800,00
R$ 8.700,00
